In [2]:
import metview as mv
from datetime import datetime, timedelta
import os



In [3]:
#
# PROBABILITY OF PRECIPITATION TYPE (%)
# ---------------------------------------
#
# Estibaliz Gascon 12/01/2017
# Revised: 02/01/2025
#
########################################

# ===== CONFIGURATION =====
# Define date range (instead of hardcoded list)
start_date = datetime(2023, 12, 11)  # Start date: 2 December 2020
end_date = datetime(2024, 3, 15)    # End date: 28 February 2021

# Generate list of dates in YYYYMMDD format
fdates = []
current_date = start_date
while current_date <= end_date:
    fdates.append(int(current_date.strftime('%Y%m%d')))
    current_date += timedelta(days=1)

# Extended forecast steps (0-168h in 3h increments, except after 144h)
lsteps = list(range(0, 145, 3))  # 0, 3, 6, ..., 144
lsteps.extend([150, 156, 162, 168])  # Add the remaining steps

# Precipitation rate thresholds (mm/h) for each type
prec_rate_threshold_rain = 0.12
prec_rate_threshold_snow = 0.04
prec_rate_threshold_wetsnow = 0.04
prec_rate_threshold_sleet = 0.1
prec_rate_threshold_freezrain = 0.05
prec_rate_threshold_freezdriz = 0.03
prec_rate_threshold_icepellets = 0.05

# Path configuration
PATH = "/ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/raw/raw_2023_2024/"
PATH_OUT_BASE = "/ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/probabilities_ptype/2023_2024/"

if not os.path.exists(PATH_OUT_BASE):
    print(f"Creating output base directory: {PATH_OUT_BASE}")
    os.makedirs(PATH_OUT_BASE, exist_ok=True)
else:
    print(f"Output base directory exists: {PATH_OUT_BASE}")

# Number of ensemble members
n_members = 51

print(f"Processing {len(fdates)} dates from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print(f"Forecast steps: {lsteps}")
print(f"Total steps per date: {len(lsteps)}")



Output base directory exists: /ec/vol/enscal/VERIF_PrecipType/VERIF_ptype_all/probabilities_ptype/2023_2024/
Processing 96 dates from 2023-12-11 to 2024-03-15
Forecast steps: [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36, 39, 42, 45, 48, 51, 54, 57, 60, 63, 66, 69, 72, 75, 78, 81, 84, 87, 90, 93, 96, 99, 102, 105, 108, 111, 114, 117, 120, 123, 126, 129, 132, 135, 138, 141, 144, 150, 156, 162, 168]
Total steps per date: 53


In [ ]:
# ===== MAIN PROCESSING LOOP =====
for fdate in fdates:
    
    PATH_OUT = PATH_OUT_BASE + str(fdate) + "_0/"
    mv.shell("mkdir -p " + str(PATH_OUT))

    print(f"\nProcessing date: {fdate}")
    
    # Read the input files
    try:
        Prectype_raw_world = mv.read(
            str(PATH) + "Prectype_raw_VERIF_" + str(fdate) + ".grib"
        )
        total_prec_world = mv.read(
            str(PATH) + "total_prec_VERIF_" + str(fdate) + ".grib"
        )
    except Exception as e:
        print(f"Error reading files for {fdate}: {e}")
        continue

    # Calculate probabilities of precipitation type
    print("  Calculating probabilities...")

    # Process each forecast step
    for step in lsteps:
        print(f"    Step: {step:3d}h", end='\r')

        try:
            # Read data for specific step
            Prectype_raw = mv.read(data=Prectype_raw_world, step=step)
            prec = mv.read(data=total_prec_world, step=step)

            # Apply precipitation rate thresholds
            pthres_rain = prec >= prec_rate_threshold_rain
            pthres_snow = prec >= prec_rate_threshold_snow
            pthres_wetsnow = prec >= prec_rate_threshold_wetsnow
            pthres_sleet = prec >= prec_rate_threshold_sleet
            pthres_freezrain = prec >= prec_rate_threshold_freezrain
            pthres_freezdriz = prec >= prec_rate_threshold_freezdriz
            pthres_icepellets = prec >= prec_rate_threshold_icepellets

            # Calculate probability for each precipitation type
            # Rain (code = 1)
            rain = (Prectype_raw == 1) * pthres_rain
            rain = mv.sum(rain)
            perc_rain = rain / n_members * 100

            # Freezing rain (code = 3)
            freezrain = (Prectype_raw == 3) * pthres_freezrain
            freezrain = mv.sum(freezrain)
            perc_freezrain = freezrain / n_members * 100

            # Snow (code = 5)
            snow = (Prectype_raw == 5) * pthres_snow
            snow = mv.sum(snow)
            perc_snow = snow / n_members * 100

            # Wet snow (code = 6)
            wetsnow = (Prectype_raw == 6) * pthres_wetsnow
            wetsnow = mv.sum(wetsnow)
            perc_wetsnow = wetsnow / n_members * 100

            # Sleet (code = 7)
            sleet = (Prectype_raw == 7) * pthres_sleet
            sleet = mv.sum(sleet)
            perc_sleet = sleet / n_members * 100

            # Ice pellets (code = 8)
            icepellets = (Prectype_raw == 8) * pthres_icepellets
            icepellets = mv.sum(icepellets)
            perc_icepellets = icepellets / n_members * 100

            # Freezing drizzle (code = 12)
            freezdrizzle = (Prectype_raw == 12) * pthres_freezdriz
            freezdrizzle = mv.sum(freezdrizzle)
            perc_freezdrizzle = freezdrizzle / n_members * 100

            # Write output files
            mv.write(str(PATH_OUT) + "perc_rain_" + str(step) + ".grib", perc_rain)
            mv.write(str(PATH_OUT) + "perc_snow_" + str(step) + ".grib", perc_snow)
            mv.write(str(PATH_OUT) + "perc_wetsnow_" + str(step) + ".grib", perc_wetsnow)
            mv.write(str(PATH_OUT) + "perc_sleet_" + str(step) + ".grib", perc_sleet)
            mv.write(str(PATH_OUT) + "perc_freezrain_" + str(step) + ".grib", perc_freezrain)
            mv.write(str(PATH_OUT) + "perc_freezdrizzle_" + str(step) + ".grib", perc_freezdrizzle)
            mv.write(str(PATH_OUT) + "perc_icepellets_" + str(step) + ".grib", perc_icepellets)

        except Exception as e:
            print(f"\n    Error processing step {step}: {e}")
            continue
    
    print(f"    Completed all {len(lsteps)} steps for {fdate}")

print("\n" + "="*50)
print("Processing complete!")


Processing date: 20231211
  Calculating probabilities...
    Step:  18h